# Public repository note

This notebook is an output-cleared code copy. It requires locally authorised data and is not runnable from the public repository alone. Green Street raw data, intermediate files, derived aggregates and outputs are not distributed.


# 06 Source Disagreement and H1 Model Refinement

This notebook responds to two supervisory priorities. First, it unpacks where Green Street and OpenLocal vacancy observations diverge and creates a reproducible case-review sample. Second, it refines the H1 residential-exposure test by estimating both absolute and normalised commuter-exposure specifications.

The source-disagreement section is diagnostic. It does not assume that either source is a reliability benchmark. Green Street has confirmed its operational vacancy definition, survey process, High Street denominator and methodological consistency across 2019-2025. The comparison therefore evaluates expected differences between two complementary observation systems rather than attempting to prove that they measure an identical vacancy rate.

## 1.1 Confirmed Source-Method Difference

Green Street combines approximately six-monthly field visits with office-based research and uses a consumer-facing trading rule. A unit remains vacant until a consumer can enter and make a purchase, even when it is under offer or preparing to reopen. Its High Street denominator contains eligible live retail businesses and vacant units, while excluding Non-Retail, Miscellaneous and demolished premises. Green Street confirmed that this methodology remained consistent between 2019 and 2025; the 2024 change affected file structure rather than vacancy classification.

OpenLocal is derived from administrative commercial-property records. Its occupation-state field indicates the recorded occupation of a rateable hereditament and can therefore identify commercially occupied space that is not visibly open to consumers, including storage or non-public production uses. Conversely, administrative status may lag a visible business closure.

Agreement is consequently informative but is not a reliability pass/fail test. Close agreement indicates convergence between consumer-facing and administrative evidence. Disagreement is investigated as a possible consequence of property-universe coverage, survey timing, non-public commercial use, administrative lag or spatial aggregation.

## 1. Analytical Decisions

- **Absolute flow:** retains the scale of commuting activity linked to selected workplace MSOAs.
- **Selected-flow share:** measures the proportion of an origin MSOA's outbound commuters travelling to selected workplaces.
- **Normalised shock exposure:** divides shock-weighted selected flows by all outbound commuters from the origin MSOA.
- **Controlled H1 models:** adjust for baseline retail stock, value, floor area, vacancy and distance to the nearest selected workplace MSOA.
- **Spatial diagnostic:** Moran's I is applied to residuals from the controlled normalised-exposure models. It diagnoses remaining spatial structure; it does not automatically justify a spatial-lag model.

In [ ]:
from pathlib import Path
import os
import warnings

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from matplotlib.lines import Line2D
from scipy import stats
from shapely import wkb

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

BASE = Path(os.environ.get("DISSERTATION_WORKSPACE", Path.cwd().resolve()))
MAP_DIR = BASE / "outputs" / "restricted_mapping_and_spatial_analysis"
ORIGIN_DIR = BASE / "outputs" / "restricted_msoa_origin_exposure_analysis"
OUT_DIR = BASE / "outputs" / "restricted_source_disagreement_h1_refinement"
FIG_DIR = OUT_DIR / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

PALETTE = {
    "blue": "#4f8fb8",
    "blue_dark": "#245b78",
    "orange": "#d98745",
    "orange_dark": "#965029",
    "green": "#6f9b88",
    "red": "#bd6a61",
    "grey": "#7f878b",
    "ink": "#273238",
}

files = {
    "source_panel": MAP_DIR / "source_validation_msoa_year_panel.csv",
    "msoa_exposure": ORIGIN_DIR / "h1_origin_msoa_exposure.csv",
    "msoa_change": ORIGIN_DIR / "openlocal_msoa_2019_to_post_mean_change_indicators.csv",
    "msoa_yearly_change": ORIGIN_DIR / "openlocal_msoa_yearly_change_indicators.csv",
    "msoa_boundaries": ORIGIN_DIR / "london_msoa_2021_boundaries.geojson",
    "workplace_targets": ORIGIN_DIR / "h1_selected_workplace_msoa_targets.csv",
    "od": BASE / "ODWP01EW_MSOA.csv",
    "openlocal_raw": BASE / "2025-12-31-shuting-yang-greenstreet-retail.parquet",
    "lad_boundaries": BASE / "Local_Authority_Districts_May_2024_Boundaries_UK_BGC_3503156029110784919.geojson",
    "greenstreet_historical": BASE / "Greater London Export (2019 to 2026).csv",
    "greenstreet_current": BASE / "Greater_London_POI_Churn_2026-07-01-1110.csv",
}
missing = [name for name, path in files.items() if not path.exists()]
if missing:
    raise FileNotFoundError(f"Missing required inputs: {missing}")

for name, path in files.items():
    print(f"{name:20s} {path.name}")

## 2. Green Street and OpenLocal Vacancy Disagreement Audit

In [ ]:
source = pd.read_csv(files["source_panel"])
source = source[source["year"].isin([2019, 2023, 2024, 2025])].copy()
source = source.dropna(subset=["gs_weighted_vacancy", "vacancy_proxy"])
source["vacancy_gap"] = source["gs_weighted_vacancy"] - source["vacancy_proxy"]
source["absolute_gap"] = source["vacancy_gap"].abs()

def disagreement_class(gap):
    if abs(gap) <= 0.02:
        return "Close agreement (<=2 pp)"
    if gap >= 0.10:
        return "Green Street >=10 pp higher"
    if gap <= -0.10:
        return "OpenLocal >=10 pp higher"
    return "Moderate difference (2-10 pp)"

source["comparison_class"] = source["vacancy_gap"].map(disagreement_class)
class_order = [
    "Close agreement (<=2 pp)",
    "Moderate difference (2-10 pp)",
    "Green Street >=10 pp higher",
    "OpenLocal >=10 pp higher",
]
source["comparison_class"] = pd.Categorical(source["comparison_class"], class_order, ordered=True)

audit_summary = (
    source.groupby("comparison_class", observed=False)
    .agg(
        msoa_years=("MSOA21CD", "size"),
        msoas=("MSOA21CD", "nunique"),
        median_gap=("vacancy_gap", "median"),
        median_absolute_gap=("absolute_gap", "median"),
    )
    .reset_index()
)
audit_summary.to_csv(OUT_DIR / "source_disagreement_class_summary.csv", index=False)

review_parts = []
for category in class_order:
    subset = source[source["comparison_class"].astype(str).eq(category)].copy()
    subset = subset.sort_values(["absolute_gap", "gs_units", "retail_units"], ascending=[False, False, False])
    review_parts.append(subset.head(8))
case_review = pd.concat(review_parts, ignore_index=True)
keep = [
    "MSOA21CD", "MSOA21NM", "year", "comparison_class",
    "gs_weighted_vacancy", "vacancy_proxy", "vacancy_gap",
    "gs_properties", "gs_units", "retail_units",
]
case_review = case_review[keep].copy()
case_review["manual_review_status"] = "Method classified; premises review pending"
case_review["Green_Street_method_note"] = "Consumer-facing field/office observation; vacant until customers can enter and purchase"
case_review["OpenLocal_method_note"] = "Administrative hereditament occupation proxy; may include occupied non-public commercial use"
case_review["possible_explanation"] = ""
case_review.to_csv(OUT_DIR / "vacancy_disagreement_manual_case_review.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(11.6, 4.8), gridspec_kw={"width_ratios": [1.65, 1]})
colors = {
    class_order[0]: PALETTE["green"],
    class_order[1]: PALETTE["grey"],
    class_order[2]: PALETTE["orange"],
    class_order[3]: PALETTE["blue"],
}
for category in class_order:
    sub = source[source["comparison_class"].astype(str).eq(category)]
    axes[0].scatter(
        sub["vacancy_proxy"], sub["gs_weighted_vacancy"],
        s=18, alpha=0.58, color=colors[category], edgecolor="white", linewidth=0.2,
        label=category,
    )
lim = max(source["vacancy_proxy"].quantile(0.995), source["gs_weighted_vacancy"].quantile(0.995), 0.2)
axes[0].plot([0, lim], [0, lim], color=PALETTE["ink"], linewidth=1, linestyle="--")
axes[0].set_xlim(-0.01, lim * 1.03)
axes[0].set_ylim(-0.01, lim * 1.03)
axes[0].set_xlabel("OpenLocal occupation-based vacancy proxy")
axes[0].set_ylabel("Green Street weighted vacancy")
axes[0].set_title("A. MSOA-year vacancy comparison")
axes[0].legend(frameon=False, fontsize=7.3, loc="upper left")
axes[0].grid(alpha=0.18)

count_data = audit_summary.set_index("comparison_class").reindex(class_order)
axes[1].barh(
    np.arange(len(class_order)), count_data["msoa_years"],
    color=[colors[c] for c in class_order], alpha=0.88,
)
axes[1].set_yticks(np.arange(len(class_order)), [
    "Close agreement", "Moderate difference", "GS >=10 pp higher", "OL >=10 pp higher"
])
axes[1].invert_yaxis()
axes[1].set_xlabel("MSOA-year observations")
axes[1].set_title("B. Review strata")
axes[1].grid(axis="x", alpha=0.18)
for i, value in enumerate(count_data["msoa_years"]):
    axes[1].text(value + max(count_data["msoa_years"]) * 0.015, i, f"{int(value)}", va="center", fontsize=8)

fig.suptitle("Green Street and OpenLocal Vacancy Disagreement Audit", x=0.02, ha="left", fontsize=14)
fig.tight_layout(rect=[0, 0, 1, 0.93])
fig.savefig(FIG_DIR / "fig_01_source_disagreement_audit.png", bbox_inches="tight", pad_inches=0.04, dpi=300)
plt.show()

display(audit_summary)
display(case_review.head(12))

### Interpretation boundary

The thresholds above are review rules, not definitions of measurement error. Green Street's confirmed consumer-facing trading rule differs from OpenLocal's administrative occupation concept, so the sign and size of each gap cannot identify which source is "correct". The case list instead identifies where postcode- or premises-level review can test explanations such as non-public commercial use, survey timing, administrative lag and differences in the monitored property universe.

## 2.1 Postcode-Level Case Review

In [ ]:
# Select two reproducible MSOA-year cases from each disagreement stratum.
selected_cases = (
    case_review.assign(absolute_gap=case_review["vacancy_gap"].abs())
    .sort_values(["comparison_class", "absolute_gap"], ascending=[True, False])
    .groupby("comparison_class", observed=True, group_keys=False)
    .head(2)
    .reset_index(drop=True)
)
selected_cases["case_id"] = [f"CASE_{i:02d}" for i in range(1, len(selected_cases) + 1)]
selected_cases.to_csv(OUT_DIR / "vacancy_case_review_selected_msoa_years.csv", index=False)

case_msoa = gpd.read_file(files["msoa_boundaries"]).to_crs("EPSG:27700")
case_msoa = case_msoa[case_msoa["MSOA21CD"].isin(selected_cases["MSOA21CD"])][
    ["MSOA21CD", "MSOA21NM", "geometry"]
].copy()

def normalise_postcode(series):
    return series.astype("string").str.upper().str.replace(r"\s+", "", regex=True).str.strip()

# OpenLocal is observed quarterly. Aggregate all four snapshots within each selected year.
selected_years = sorted(selected_cases["year"].astype(int).unique())
quarterly_periods = [f"{year}-{month:02d}-01" for year in selected_years for month in [1, 4, 7, 10]]
ol_case = pd.read_parquet(
    files["openlocal_raw"],
    columns=["period", "uarn", "occupation_state", "postcode_id", "geometry", "category_group"],
    filters=[("category_group", "==", "RETAIL"), ("period", "in", quarterly_periods)],
)
ol_case["year"] = pd.to_datetime(ol_case["period"]).dt.year
ol_case["geometry"] = ol_case["geometry"].map(
    lambda value: wkb.loads(bytes.fromhex(str(value))) if pd.notna(value) else None
)
ol_case = ol_case.dropna(subset=["geometry", "postcode_id", "uarn"]).copy()
ol_case = gpd.GeoDataFrame(ol_case, geometry="geometry", crs="EPSG:4326").to_crs("EPSG:27700")
ol_case = gpd.sjoin(ol_case, case_msoa, how="inner", predicate="within").drop(columns="index_right", errors="ignore")
ol_case = ol_case.merge(
    selected_cases[["case_id", "MSOA21CD", "year", "comparison_class"]],
    on=["MSOA21CD", "year"], how="inner",
)
ol_case["postcode_key"] = normalise_postcode(ol_case["postcode_id"])
ol_postcode = (
    ol_case.groupby(["case_id", "MSOA21CD", "MSOA21NM", "year", "comparison_class", "postcode_key"], observed=True)
    .agg(
        ol_record_observations=("uarn", "size"),
        ol_units=("uarn", "nunique"),
        ol_classified_observations=("occupation_state", lambda s: s.astype(str).str.upper().isin(["OCCUPIED", "VACANT"]).sum()),
        ol_vacant_observations=("occupation_state", lambda s: s.astype(str).str.upper().eq("VACANT").sum()),
    )
    .reset_index()
)
# The derived proxy uses only records with a reported OCCUPIED or VACANT
# state. Unclassified records are not treated as occupied by default.
ol_postcode["ol_postcode_vacancy"] = ol_postcode["ol_vacant_observations"] / ol_postcode["ol_classified_observations"].replace(0, np.nan)
ol_postcode["ol_classification_completeness"] = ol_postcode["ol_classified_observations"] / ol_postcode["ol_record_observations"].replace(0, np.nan)

# Green Street is normally revisited half-yearly. Reconstruct June and December recorded states.
gs = pd.read_csv(files["greenstreet_historical"], low_memory=False)
gs["CreatedDate"] = pd.to_datetime(gs["CreatedDate"], errors="coerce")
gs["ClosedDate"] = pd.to_datetime(gs["ClosedDate"], errors="coerce")
gs["Latitude"] = pd.to_numeric(gs["Latitude"], errors="coerce")
gs["Longitude"] = pd.to_numeric(gs["Longitude"], errors="coerce")
gs["PremiseId"] = gs["PremiseId"].astype("string").str.replace(r"\.0$", "", regex=True)
gs["Tenant ID"] = gs["Tenant ID"].astype("string").str.replace(r"\.0$", "", regex=True)

gs_category = pd.read_csv(
    files["greenstreet_current"], usecols=["SUBCATEGORY", "CATEGORY"], low_memory=False
).dropna()
gs_crosswalk = (
    gs_category.groupby("SUBCATEGORY")["CATEGORY"]
    .agg(lambda values: values.mode().iloc[0])
    .rename("category")
    .reset_index()
    .rename(columns={"SUBCATEGORY": "SubCategory"})
)
gs = gs.merge(gs_crosswalk, on="SubCategory", how="left")
gs["record_type"] = np.select(
    [
        gs["BusinessType"].isin(["Independent", "Multiple"]),
        gs["SubCategory"].eq("Vacant Properties"),
    ],
    ["Retail business", "Vacant"],
    default="Excluded",
)
gs.loc[gs["category"].isin(["Non-Retail", "Miscellaneous"]), "record_type"] = "Excluded"
gs = gs[gs["record_type"].isin(["Retail business", "Vacant"])].dropna(subset=["Latitude", "Longitude", "PostCode"]).copy()
gs = gpd.GeoDataFrame(
    gs, geometry=gpd.points_from_xy(gs["Longitude"], gs["Latitude"]), crs="EPSG:4326"
).to_crs("EPSG:27700")
gs = gpd.sjoin(gs, case_msoa, how="inner", predicate="within").drop(columns="index_right", errors="ignore")
gs["postcode_key"] = normalise_postcode(gs["PostCode"])

gs_snapshots = []
for year in selected_years:
    for snapshot in [pd.Timestamp(year, 6, 30), pd.Timestamp(year, 12, 31)]:
        active = gs[
            gs["CreatedDate"].le(snapshot)
            & (gs["ClosedDate"].isna() | gs["ClosedDate"].gt(snapshot))
        ].copy()
        active = active.sort_values(["PremiseId", "CreatedDate", "Tenant ID"]).drop_duplicates("PremiseId", keep="last")
        active["year"] = year
        active["snapshot"] = snapshot
        gs_snapshots.append(active)
gs_case = pd.concat(gs_snapshots, ignore_index=True)
gs_case = gs_case.merge(
    selected_cases[["case_id", "MSOA21CD", "year", "comparison_class"]],
    on=["MSOA21CD", "year"], how="inner",
)
gs_postcode = (
    gs_case.groupby(["case_id", "MSOA21CD", "MSOA21NM", "year", "comparison_class", "postcode_key"], observed=True)
    .agg(
        gs_record_observations=("PremiseId", "size"),
        gs_premises=("PremiseId", "nunique"),
        gs_vacant_observations=("record_type", lambda s: s.eq("Vacant").sum()),
    )
    .reset_index()
)
gs_postcode["gs_postcode_vacancy"] = gs_postcode["gs_vacant_observations"] / gs_postcode["gs_record_observations"].replace(0, np.nan)

postcode_review = gs_postcode.merge(
    ol_postcode,
    on=["case_id", "MSOA21CD", "MSOA21NM", "year", "comparison_class", "postcode_key"],
    how="outer",
)
postcode_review["source_coverage"] = np.select(
    [
        postcode_review["gs_record_observations"].notna() & postcode_review["ol_record_observations"].notna(),
        postcode_review["gs_record_observations"].notna(),
    ],
    ["Both sources", "Green Street only"],
    default="OpenLocal only",
)
postcode_review["postcode_vacancy_gap"] = postcode_review["gs_postcode_vacancy"] - postcode_review["ol_postcode_vacancy"]

def postcode_explanation(row):
    if row["source_coverage"] != "Both sources":
        return "Coverage/universe difference"
    if abs(row["postcode_vacancy_gap"]) <= 0.10:
        return "Close postcode-level agreement"
    if row["postcode_vacancy_gap"] >= 0.25:
        return "Consumer-facing vacancy / administrative occupation candidate"
    if row["postcode_vacancy_gap"] <= -0.25:
        return "Administrative vacancy / observed trading candidate"
    return "Moderate timing, unit-definition or classification difference"

postcode_review["diagnostic_class"] = postcode_review.apply(postcode_explanation, axis=1)
postcode_review.to_csv(OUT_DIR / "vacancy_case_review_postcode_diagnostics_restricted.csv", index=False)

case_summary = (
    postcode_review.groupby(["case_id", "MSOA21CD", "MSOA21NM", "year", "comparison_class"], observed=True)
    .agg(
        postcodes=("postcode_key", "nunique"),
        shared_postcodes=("source_coverage", lambda s: s.eq("Both sources").sum()),
        greenstreet_only_postcodes=("source_coverage", lambda s: s.eq("Green Street only").sum()),
        openlocal_only_postcodes=("source_coverage", lambda s: s.eq("OpenLocal only").sum()),
        median_shared_gap=("postcode_vacancy_gap", "median"),
    )
    .reset_index()
)
case_summary.to_csv(OUT_DIR / "vacancy_case_review_case_summary.csv", index=False)

manual_candidates = (
    postcode_review.assign(abs_postcode_gap=postcode_review["postcode_vacancy_gap"].abs())
    .sort_values(["case_id", "abs_postcode_gap"], ascending=[True, False])
    .groupby("case_id", group_keys=False)
    .head(3)
)
manual_candidates.to_csv(OUT_DIR / "vacancy_case_review_manual_candidates_restricted.csv", index=False)

diagnostic_order = [
    "Close postcode-level agreement",
    "Moderate timing, unit-definition or classification difference",
    "Consumer-facing vacancy / administrative occupation candidate",
    "Administrative vacancy / observed trading candidate",
    "Coverage/universe difference",
]
diagnostic_counts = (
    postcode_review.groupby(["case_id", "diagnostic_class"]).size().unstack(fill_value=0).reindex(columns=diagnostic_order, fill_value=0)
)
fig, axes = plt.subplots(1, 2, figsize=(11.6, 4.8), gridspec_kw={"width_ratios": [1.25, 1]})
diagnostic_counts.plot(kind="bar", stacked=True, ax=axes[0], color=[PALETTE["green"], PALETTE["grey"], PALETTE["orange"], PALETTE["blue"], PALETTE["red"]])
axes[0].set_xlabel("Anonymised MSOA-year case")
axes[0].set_ylabel("Postcodes")
axes[0].set_title("A. Postcode diagnostic composition")
axes[0].tick_params(axis="x", rotation=0)
axes[0].legend(frameon=False, fontsize=6.8, loc="upper left")

shared = postcode_review[postcode_review["source_coverage"].eq("Both sources")].dropna(
    subset=["gs_postcode_vacancy", "ol_postcode_vacancy"]
)
axes[1].scatter(shared["ol_postcode_vacancy"], shared["gs_postcode_vacancy"], s=18, alpha=0.42, color=PALETTE["blue_dark"], edgecolor="white", linewidth=0.2)
axes[1].plot([0, 1], [0, 1], linestyle="--", color=PALETTE["ink"], linewidth=0.8)
axes[1].set_xlim(-0.03, 1.03)
axes[1].set_ylim(-0.03, 1.03)
axes[1].set_xlabel("OpenLocal occupation-based vacancy proxy")
axes[1].set_ylabel("Green Street recorded postcode vacancy")
axes[1].set_title("B. Shared-postcode comparison")
axes[1].grid(alpha=0.15)
fig.suptitle("Vacancy Difference Case Review: Postcode-Level Evidence", x=0.02, ha="left", fontsize=14)
fig.tight_layout(rect=[0, 0, 1, 0.93])
fig.savefig(FIG_DIR / "fig_01c_vacancy_postcode_case_review.png", bbox_inches="tight", pad_inches=0.04, dpi=300)
plt.show()

display(case_summary)
display(manual_candidates.drop(columns=["postcode_key"], errors="ignore").head(12))

### Case-review interpretation

The postcode drill-down aligns source-specific observation cycles rather than assuming exact same-day measurement. OpenLocal contributes four quarterly snapshots per year; Green Street contributes reconstructed June and December recorded states. The comparison therefore identifies mechanisms consistent with the observed disagreement, but it does not adjudicate which individual record is correct. Full postcodes remain in restricted outputs only.

## 2.2 OpenLocal Quarterly Vacancy-State Audit

In [ ]:
# The annual MSOA comparison can conceal an abrupt change within a year.
# Audit the original quarterly snapshots before interpreting disagreement.
audit_lads = set(source["MSOA21NM"].str.replace(r" \d{3}$", "", regex=True))
raw_ol = pd.read_parquet(
    files["openlocal_raw"],
    columns=["period", "geocode_name", "category_group", "occupation_state", "uarn"],
)
raw_ol = raw_ol[
    raw_ol["category_group"].astype(str).str.upper().eq("RETAIL")
    & raw_ol["geocode_name"].isin(audit_lads)
].copy()
raw_ol["period"] = pd.to_datetime(raw_ol["period"])
raw_ol["is_occupied"] = raw_ol["occupation_state"].astype(str).str.upper().eq("OCCUPIED")
raw_ol["is_vacant"] = raw_ol["occupation_state"].astype(str).str.upper().eq("VACANT")

quarterly_status = (
    raw_ol.groupby(["geocode_name", "period"], as_index=False)
    .agg(
        records=("uarn", "size"),
        unique_units=("uarn", "nunique"),
        occupied_records=("is_occupied", "sum"),
        vacant_records=("is_vacant", "sum"),
    )
    .sort_values(["geocode_name", "period"])
)
quarterly_status["classified_records"] = quarterly_status["occupied_records"] + quarterly_status["vacant_records"]
quarterly_status["classification_completeness"] = quarterly_status["classified_records"] / quarterly_status["records"].replace(0, np.nan)
# Reproduce the existing OpenLocal proxy: VACANT records divided by all retail records.
quarterly_status["vacancy_proxy"] = quarterly_status["vacant_records"] / quarterly_status["records"].replace(0, np.nan)
quarterly_status["quarterly_vacancy_change"] = quarterly_status.groupby("geocode_name")["vacancy_proxy"].diff()

lad_anomaly_summary = (
    quarterly_status.groupby("geocode_name", as_index=False)
    .agg(
        max_vacancy_proxy=("vacancy_proxy", "max"),
        max_absolute_quarterly_jump=("quarterly_vacancy_change", lambda s: s.abs().max()),
        median_classification_completeness=("classification_completeness", "median"),
        minimum_classification_completeness=("classification_completeness", "min"),
    )
)
lad_anomaly_summary["severe_temporal_discontinuity"] = (
    lad_anomaly_summary["max_absolute_quarterly_jump"].ge(0.30)
    | lad_anomaly_summary["max_vacancy_proxy"].ge(0.50)
)
anomalous_vacancy_lads = set(
    lad_anomaly_summary.loc[lad_anomaly_summary["severe_temporal_discontinuity"], "geocode_name"]
)
source["origin_lad_name"] = source["MSOA21NM"].str.replace(r" \d{3}$", "", regex=True)
source_comparison_rows = []
for sample_label, comparison_sample in {
    "All matched MSOA-years": source,
    "Exclude severe temporal anomaly LADs": source[~source["origin_lad_name"].isin(anomalous_vacancy_lads)],
}.items():
    source_comparison_rows.append({
        "sample": sample_label,
        "n_msoa_years": len(comparison_sample),
        "pearson_r": stats.pearsonr(comparison_sample["gs_weighted_vacancy"], comparison_sample["vacancy_proxy"]).statistic,
        "spearman_r": stats.spearmanr(comparison_sample["gs_weighted_vacancy"], comparison_sample["vacancy_proxy"]).statistic,
        "median_absolute_gap": comparison_sample["vacancy_gap"].abs().median(),
    })
source_comparison_sensitivity = pd.DataFrame(source_comparison_rows)
quarterly_status.to_csv(OUT_DIR / "openlocal_quarterly_vacancy_status_audit.csv", index=False)
lad_anomaly_summary.to_csv(OUT_DIR / "openlocal_vacancy_anomaly_by_lad.csv", index=False)
source_comparison_sensitivity.to_csv(OUT_DIR / "source_alignment_anomaly_sensitivity.csv", index=False)

heat = quarterly_status.pivot(index="geocode_name", columns="period", values="vacancy_proxy")
heat = heat.loc[heat.max(axis=1).sort_values(ascending=False).index]
fig, ax = plt.subplots(figsize=(12.0, max(5.2, 0.27 * len(heat))))
image = ax.imshow(heat.to_numpy(), aspect="auto", cmap="YlOrRd", vmin=0, vmax=min(1, heat.quantile(0.99).max()))
ax.set_yticks(np.arange(len(heat)), heat.index, fontsize=7)
tick_positions = np.arange(0, len(heat.columns), 2)
ax.set_xticks(tick_positions, [pd.Timestamp(heat.columns[i]).strftime("%Y Q") + str((pd.Timestamp(heat.columns[i]).month - 1) // 3 + 1) for i in tick_positions], rotation=45, ha="right", fontsize=7)
ax.set_title("OpenLocal Quarterly Occupation-Based Vacancy Proxy Audit", loc="left", fontsize=13)
cb = fig.colorbar(image, ax=ax, shrink=0.58, pad=0.015)
cb.set_label("VACANT records / all retail records", fontsize=8)
cb.ax.tick_params(labelsize=7)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig_01b_openlocal_quarterly_vacancy_audit.png", bbox_inches="tight", pad_inches=0.04, dpi=300)
plt.show()

print("LADs flagged for severe temporal discontinuity:", sorted(anomalous_vacancy_lads))
display(lad_anomaly_summary.sort_values("max_absolute_quarterly_jump", ascending=False).head(10))
display(source_comparison_sensitivity)
del raw_ol

## 3. Construct Absolute and Normalised Residential Exposure

In [ ]:
exposure = pd.read_csv(files["msoa_exposure"])
selected_origin_codes = set(exposure["origin_msoa"])
workplace_targets_for_distance = pd.read_csv(files["workplace_targets"])
selected_workplace_codes = set(workplace_targets_for_distance["workplace_msoa"])
distance_boundaries = gpd.read_file(files["msoa_boundaries"]).to_crs("EPSG:27700")
centroid_xy = distance_boundaries[["MSOA21CD", "geometry"]].copy()
centroid_xy["x"] = centroid_xy.geometry.centroid.x
centroid_xy["y"] = centroid_xy.geometry.centroid.y
centroid_xy = centroid_xy.drop(columns="geometry")

od_cols = [
    "Middle layer Super Output Areas code",
    "MSOA of workplace code",
    "Place of work indicator (4 categories) code",
    "Count",
]
od = pd.read_csv(files["od"], usecols=od_cols)
od = od.rename(columns={
    "Middle layer Super Output Areas code": "origin_msoa",
    "MSOA of workplace code": "workplace_msoa",
    "Place of work indicator (4 categories) code": "workplace_indicator_code",
    "Count": "commuters",
})
od["commuters"] = pd.to_numeric(od["commuters"], errors="coerce").fillna(0)
outbound = od[
    od["workplace_indicator_code"].eq(3)
    & od["origin_msoa"].isin(selected_origin_codes)
    & od["workplace_msoa"].astype(str).str.match(r"^[EW]02", na=False)
].groupby("origin_msoa", as_index=False)["commuters"].sum()
outbound = outbound.rename(columns={"commuters": "total_outbound_commuters"})

# Gavin's distance mechanism: calculate the average straight-line distance
# actually travelled across selected origin-destination pairs, weighted by flow.
od_selected = od[
    od["workplace_indicator_code"].eq(3)
    & od["origin_msoa"].isin(selected_origin_codes)
    & od["workplace_msoa"].isin(selected_workplace_codes)
].copy()
origin_xy = centroid_xy.rename(columns={"MSOA21CD": "origin_msoa", "x": "origin_x", "y": "origin_y"})
workplace_xy = centroid_xy.rename(columns={"MSOA21CD": "workplace_msoa", "x": "workplace_x", "y": "workplace_y"})
od_selected = od_selected.merge(origin_xy, on="origin_msoa", how="left").merge(workplace_xy, on="workplace_msoa", how="left")
od_selected["od_distance_km"] = np.sqrt(
    (od_selected["origin_x"] - od_selected["workplace_x"]) ** 2
    + (od_selected["origin_y"] - od_selected["workplace_y"]) ** 2
) / 1000
od_selected["flow_distance_product"] = od_selected["commuters"] * od_selected["od_distance_km"]
weighted_distance = (
    od_selected.groupby("origin_msoa", as_index=False)
    .agg(
        selected_commuters_distance_denominator=("commuters", "sum"),
        flow_distance_sum=("flow_distance_product", "sum"),
    )
)
weighted_distance["flow_weighted_commute_distance_km"] = (
    weighted_distance["flow_distance_sum"]
    / weighted_distance["selected_commuters_distance_denominator"].replace(0, np.nan)
)
del od

exposure = exposure.merge(outbound, on="origin_msoa", how="left").merge(
    weighted_distance[["origin_msoa", "flow_weighted_commute_distance_km"]],
    on="origin_msoa",
    how="left",
)
exposure["absolute_selected_flow"] = exposure["commuters_to_selected_workplaces"]
exposure["absolute_shock_exposure_per_1000"] = exposure["od_weighted_msoa_exposure_sum"] / 1000
exposure["selected_flow_share"] = (
    exposure["commuters_to_selected_workplaces"] / exposure["total_outbound_commuters"].replace(0, np.nan)
)
exposure["normalised_shock_exposure_per_1000_outbound"] = (
    exposure["od_weighted_msoa_exposure_sum"] / exposure["total_outbound_commuters"].replace(0, np.nan) * 1000
)
exposure["mean_shock_of_selected_flows"] = (
    exposure["od_weighted_msoa_exposure_sum"] / exposure["commuters_to_selected_workplaces"].replace(0, np.nan)
)
exposure.to_csv(OUT_DIR / "h1_origin_msoa_absolute_and_normalised_exposure.csv", index=False)

exposure_summary = exposure[[
    "absolute_selected_flow", "absolute_shock_exposure_per_1000", "selected_flow_share",
    "normalised_shock_exposure_per_1000_outbound", "mean_shock_of_selected_flows",
    "flow_weighted_commute_distance_km"
]].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95]).T
exposure_summary.to_csv(OUT_DIR / "h1_exposure_measure_summary.csv")
display(exposure_summary)

## 4. Add Baseline Retail and Distance Controls

In [ ]:
boundaries = gpd.read_file(files["msoa_boundaries"]).to_crs("EPSG:27700")
workplace_targets = pd.read_csv(files["workplace_targets"])
workplace_codes = set(workplace_targets["workplace_msoa"])
workplace_union = boundaries[boundaries["MSOA21CD"].isin(workplace_codes)].geometry.union_all()

distance = boundaries[["MSOA21CD", "geometry"]].copy()
distance["distance_to_nearest_workplace_km"] = distance.geometry.centroid.distance(workplace_union) / 1000
distance = distance.drop(columns="geometry")

change = pd.read_csv(files["msoa_change"])
model_panel = (
    change.merge(exposure, left_on="MSOA21CD", right_on="origin_msoa", how="inner")
    .merge(distance, on="MSOA21CD", how="left")
)
model_panel = model_panel[~model_panel["MSOA21CD"].isin(workplace_codes)].copy()
model_panel["origin_lad_name"] = model_panel["MSOA21NM"].str.replace(r" \d{3}$", "", regex=True)
model_panel["vacancy_temporal_anomaly_flag"] = model_panel["origin_lad_name"].isin(anomalous_vacancy_lads)
for col in ["retail_units", "total_rateable_value", "total_floor_area"]:
    model_panel[f"log_{col}_2019"] = np.log1p(model_panel[col].clip(lower=0))
model_panel.to_csv(OUT_DIR / "h1_refined_model_panel.csv", index=False)

# The cross-sectional file above is retained for maps of mean 2023-2025
# retail change. The formal origin-side regressions use annual MSOA
# observations, matching the workplace-side temporal design.
yearly_change = pd.read_csv(files["msoa_yearly_change"])
model_year_panel = (
    yearly_change.merge(exposure, left_on="MSOA21CD", right_on="origin_msoa", how="inner")
    .merge(distance, on="MSOA21CD", how="left")
)
model_year_panel = model_year_panel[
    ~model_year_panel["MSOA21CD"].isin(workplace_codes)
    & model_year_panel["year"].isin([2023, 2024, 2025])
].copy()
model_year_panel["origin_lad_name"] = model_year_panel["MSOA21NM"].str.replace(r" \\d{3}$", "", regex=True)
model_year_panel["vacancy_temporal_anomaly_flag"] = model_year_panel["origin_lad_name"].isin(anomalous_vacancy_lads)
for col in ["retail_units", "total_rateable_value", "total_floor_area"]:
    model_year_panel[f"log_{col}_2019"] = np.log1p(
        model_year_panel[f"{col}_2019"].clip(lower=0)
    )
model_year_panel.to_csv(OUT_DIR / "h1_refined_model_year_panel.csv", index=False)

# Use this annual panel in all formal RQ1b specifications below.
model_panel = model_year_panel

print("Refined origin-MSOA-year model panel:", len(model_panel))
print("MSOAs with >=20 baseline retail units:", model_panel.loc[model_panel["retail_units_2019"].ge(20), "MSOA21CD"].nunique())
display(model_panel[[
    "MSOA21CD", "MSOA21NM", "year", "absolute_shock_exposure_per_1000",
    "normalised_shock_exposure_per_1000_outbound", "selected_flow_share",
    "flow_weighted_commute_distance_km", "distance_to_nearest_workplace_km",
    "vacancy_temporal_anomaly_flag", "retail_units_2019", "retail_units"
]].head())

## 5. Refined H1 Models

In [ ]:
outcomes = {
    "Retail unit count": "retail_units_pct_change_2019",
    "Total rateable value": "total_rateable_value_pct_change_2019",
    "Retail floor area": "total_floor_area_pct_change_2019",
    "OpenLocal occupation-based vacancy proxy": "vacancy_proxy_change_2019",
}
predictors = {
    "Absolute weighted flow": "absolute_shock_exposure_per_1000",
    "Normalised weighted share": "normalised_shock_exposure_per_1000_outbound",
    "Selected commuter share": "selected_flow_share",
}
controls = [
    "log_retail_units_2019",
    "log_total_rateable_value_2019",
    "log_total_floor_area_2019",
    "vacancy_proxy_2019",
    "flow_weighted_commute_distance_km",
]

def zscore(series):
    std = series.std(ddof=0)
    return (series - series.mean()) / std if std and np.isfinite(std) else series * np.nan

def fit_refined_model(
    data, outcome, predictor, specification, min_baseline_units=20
):
    cols = ["MSOA21CD", "year", outcome, predictor, "retail_units_2019"] + controls
    clean = data[cols].replace([np.inf, -np.inf], np.nan).dropna().copy()
    clean = clean[
        clean["retail_units_2019"].ge(min_baseline_units)
        & clean[predictor].gt(0)
    ].copy()
    if len(clean) < 40:
        return None, None

    lo, hi = clean[outcome].quantile([0.01, 0.99])
    clean[outcome] = clean[outcome].clip(lo, hi)
    x_cols = [predictor] + (controls if specification == "Controlled" else [])
    X = pd.DataFrame(index=clean.index)
    for col in x_cols:
        X[col] = zscore(clean[col].astype(float))
    year_effects = pd.get_dummies(
        clean["year"].astype(str), prefix="year", drop_first=True, dtype=float
    )
    X = pd.concat([X, year_effects], axis=1)
    X = sm.add_constant(X)
    model = sm.OLS(clean[outcome].astype(float), X).fit(
        cov_type="cluster", cov_kwds={"groups": clean["MSOA21CD"]}
    )
    beta = model.params[predictor]
    se = model.bse[predictor]
    row = {
        "outcome": outcome,
        "predictor": predictor,
        "specification": specification,
        "n_obs": int(model.nobs),
        "beta_standardised_exposure": beta,
        "robust_se": se,
        "ci_low": beta - 1.96 * se,
        "ci_high": beta + 1.96 * se,
        "t_value": model.tvalues[predictor],
        "p_value": model.pvalues[predictor],
        "r_squared": model.rsquared,
        "controls": (
            "; ".join(controls + ["year indicators"])
            if specification == "Controlled" else "year indicators"
        ),
    }
    residuals = clean[["MSOA21CD"]].copy()
    residuals["residual"] = model.resid
    return row, residuals

model_rows = []
primary_residuals = []
for outcome_label, outcome in outcomes.items():
    for predictor_label, predictor in predictors.items():
        for specification in ["Unadjusted", "Controlled"]:
            row, residuals = fit_refined_model(model_panel, outcome, predictor, specification)
            if row is None:
                continue
            row.update({"outcome_label": outcome_label, "predictor_label": predictor_label})
            model_rows.append(row)
            if predictor_label == "Normalised weighted share" and specification == "Controlled":
                residuals["outcome_label"] = outcome_label
                primary_residuals.append(residuals)

model_results = pd.DataFrame(model_rows)
model_results.to_csv(OUT_DIR / "h1_refined_model_results.csv", index=False)
residual_panel = pd.concat(primary_residuals, ignore_index=True) if primary_residuals else pd.DataFrame()
residual_panel.to_csv(OUT_DIR / "h1_primary_model_residuals.csv", index=False)

plot_data = model_results[model_results["specification"].eq("Controlled")].copy()
predictor_order = list(predictors)
# Vacancy is reserved for the dedicated anomaly-sensitivity chart.
outcome_order = [label for label in outcomes if label != "OpenLocal occupation-based vacancy proxy"]
fig, axes = plt.subplots(1, 3, figsize=(10.2, 3.9), sharex=False)
offsets = np.linspace(-0.18, 0.18, len(predictor_order))
predictor_colors = [PALETTE["orange"], PALETTE["blue"], PALETTE["green"]]
for ax, outcome_label in zip(axes, outcome_order):
    sub = plot_data[plot_data["outcome_label"].eq(outcome_label)]
    for offset, predictor_label, color in zip(offsets, predictor_order, predictor_colors):
        row = sub[sub["predictor_label"].eq(predictor_label)]
        if row.empty:
            continue
        row = row.iloc[0]
        ax.errorbar(
            row["beta_standardised_exposure"], offset,
            xerr=[[row["beta_standardised_exposure"] - row["ci_low"]], [row["ci_high"] - row["beta_standardised_exposure"]]],
            fmt="o", color=color, capsize=3, markersize=5,
        )
    ax.axvline(0, color="#8d9497", linewidth=0.9)
    ax.set_yticks([])
    ax.set_title(outcome_label, fontsize=10)
    ax.set_xlabel("Exposure coefficient\n(95% CI)", fontsize=8)
    ax.grid(axis="x", alpha=0.18)
handles = [Line2D([0], [0], marker="o", color="none", markerfacecolor=c, markeredgecolor=c, label=p) for p, c in zip(predictor_order, predictor_colors)]
fig.legend(handles=handles, loc="lower center", ncol=3, frameon=False, fontsize=8, bbox_to_anchor=(0.5, -0.01))
fig.suptitle("H1 Controlled Models: Absolute and Normalised Exposure Specifications", x=0.02, ha="left", fontsize=13)
fig.tight_layout(rect=[0, 0.10, 1, 0.91])
fig.savefig(FIG_DIR / "fig_02_h1_refined_exposure_coefficients.png", bbox_inches="tight", pad_inches=0.04, dpi=300)
plt.show()
display(model_results)

# Baseline-stock sensitivity. The 20-unit rule is the main specification:
# at that denominator a one-unit change is no more than five per cent.
# Thresholds of 30 and 50 test whether results depend on retaining
# comparatively small retail markets.
threshold_rows = []
threshold_outcomes = {
    label: outcome
    for label, outcome in outcomes.items()
    if label != "OpenLocal occupation-based vacancy proxy"
}
threshold_predictors = {
    "Absolute exposure": "absolute_shock_exposure_per_1000",
    "Normalised exposure": "normalised_shock_exposure_per_1000_outbound",
}
for threshold in [20, 30, 50]:
    for outcome_label, outcome in threshold_outcomes.items():
        for predictor_label, predictor in threshold_predictors.items():
            row, _ = fit_refined_model(
                model_panel,
                outcome,
                predictor,
                "Controlled",
                min_baseline_units=threshold,
            )
            if row is None:
                continue
            row.update({
                "minimum_2019_retail_units": threshold,
                "outcome_label": outcome_label,
                "predictor_label": predictor_label,
            })
            threshold_rows.append(row)

threshold_sensitivity = pd.DataFrame(threshold_rows)
threshold_sensitivity.to_csv(
    OUT_DIR / "rq1b_baseline_stock_threshold_sensitivity.csv",
    index=False,
)

threshold_colors = {
    20: "#6f9b88",
    30: "#d98745",
    50: "#245b78",
}
outcome_order = list(threshold_outcomes)
fig, axes = plt.subplots(1, 2, figsize=(10.6, 5.3), sharey=True)
for ax, predictor_label in zip(axes, threshold_predictors):
    subset = threshold_sensitivity[
        threshold_sensitivity["predictor_label"].eq(predictor_label)
    ]
    for outcome_y, outcome_label in enumerate(outcome_order):
        for offset, threshold in zip([-0.18, 0, 0.18], [20, 30, 50]):
            row = subset[
                subset["outcome_label"].eq(outcome_label)
                & subset["minimum_2019_retail_units"].eq(threshold)
            ]
            if row.empty:
                continue
            row = row.iloc[0]
            ax.errorbar(
                row["beta_standardised_exposure"],
                outcome_y + offset,
                xerr=[[
                    row["beta_standardised_exposure"] - row["ci_low"]
                ], [
                    row["ci_high"] - row["beta_standardised_exposure"]
                ]],
                fmt="o",
                color=threshold_colors[threshold],
                capsize=3,
                markersize=6,
            )
    ax.axvline(0, color="#6b7280", linewidth=1.2, linestyle="--")
    ax.set_title(predictor_label, fontsize=14)
    ax.set_xlabel("Controlled coefficient (95% CI)", fontsize=11)
    ax.grid(axis="x", alpha=0.18)
axes[0].set_yticks(range(len(outcome_order)), outcome_order, fontsize=11)
handles = [
    Line2D(
        [0], [0], marker="o", color="none",
        markerfacecolor=threshold_colors[t],
        markeredgecolor=threshold_colors[t],
        label=f"At least {t} units",
    )
    for t in [20, 30, 50]
]
fig.legend(
    handles=handles, loc="lower center", ncol=3,
    frameon=False, fontsize=10, bbox_to_anchor=(0.5, 0.01),
)
fig.suptitle(
    "RQ1b Sensitivity to the Minimum 2019 Retail-Stock Threshold",
    x=0.02, ha="left", fontsize=17,
)
fig.tight_layout(rect=[0, 0.10, 1, 0.93])
fig.savefig(
    FIG_DIR / "fig_05_rq1b_baseline_stock_threshold_sensitivity.png",
    bbox_inches="tight", pad_inches=0.08, dpi=300,
)
plt.show()
display(threshold_sensitivity[[
    "minimum_2019_retail_units", "outcome_label", "predictor_label",
    "n_obs", "beta_standardised_exposure", "ci_low", "ci_high",
    "p_value",
]])

# Test Gavin's proposed mechanism directly: does the exposure association
# change as commuting distance increases?
interaction_controls = [
    "log_retail_units_2019",
    "log_total_rateable_value_2019",
    "log_total_floor_area_2019",
    "vacancy_proxy_2019",
]

def fit_distance_interaction(data, outcome, predictor, predictor_label):
    distance_col = "flow_weighted_commute_distance_km"
    cols = ["MSOA21CD", "year", outcome, predictor, distance_col, "retail_units_2019"] + interaction_controls
    clean = data[cols].replace([np.inf, -np.inf], np.nan).dropna().copy()
    clean = clean[clean["retail_units_2019"].ge(20) & clean[predictor].gt(0)].copy()
    lo, hi = clean[outcome].quantile([0.01, 0.99])
    clean[outcome] = clean[outcome].clip(lo, hi)

    X = pd.DataFrame(index=clean.index)
    X["exposure_z"] = zscore(clean[predictor].astype(float))
    X["distance_z"] = zscore(clean[distance_col].astype(float))
    X["exposure_x_distance"] = X["exposure_z"] * X["distance_z"]
    for col in interaction_controls:
        X[col] = zscore(clean[col].astype(float))
    year_effects = pd.get_dummies(
        clean["year"].astype(str), prefix="year", drop_first=True, dtype=float
    )
    X = pd.concat([X, year_effects], axis=1)
    X = sm.add_constant(X)
    model = sm.OLS(clean[outcome].astype(float), X).fit(
        cov_type="cluster", cov_kwds={"groups": clean["MSOA21CD"]}
    )
    row = {
        "outcome": outcome,
        "predictor_label": predictor_label,
        "n_obs": int(model.nobs),
        "exposure_beta": model.params["exposure_z"],
        "exposure_p": model.pvalues["exposure_z"],
        "distance_beta": model.params["distance_z"],
        "distance_p": model.pvalues["distance_z"],
        "interaction_beta": model.params["exposure_x_distance"],
        "interaction_se": model.bse["exposure_x_distance"],
        "interaction_ci_low": model.params["exposure_x_distance"] - 1.96 * model.bse["exposure_x_distance"],
        "interaction_ci_high": model.params["exposure_x_distance"] + 1.96 * model.bse["exposure_x_distance"],
        "interaction_p": model.pvalues["exposure_x_distance"],
        "r_squared": model.rsquared,
    }
    residuals = clean[["MSOA21CD"]].copy()
    residuals["residual"] = model.resid
    return row, residuals

interaction_rows = []
interaction_primary_residuals = []
interaction_predictors = {
    "Absolute weighted flow": "absolute_shock_exposure_per_1000",
    "Normalised weighted share": "normalised_shock_exposure_per_1000_outbound",
}
for outcome_label, outcome in outcomes.items():
    for predictor_label, predictor in interaction_predictors.items():
        row, residuals = fit_distance_interaction(model_panel, outcome, predictor, predictor_label)
        row["outcome_label"] = outcome_label
        interaction_rows.append(row)
        if predictor_label == "Normalised weighted share":
            residuals["outcome_label"] = outcome_label
            interaction_primary_residuals.append(residuals)

interaction_results = pd.DataFrame(interaction_rows)
interaction_results.to_csv(OUT_DIR / "h1_exposure_distance_interaction_results.csv", index=False)
residual_panel = pd.concat(interaction_primary_residuals, ignore_index=True)
residual_panel.to_csv(OUT_DIR / "h1_primary_interaction_model_residuals.csv", index=False)

# The occupation-based vacancy proxy is shown only in the dedicated
# anomaly-sensitivity figure below, where classification changes can be
# inspected rather than mistaken for a direct visible-vacancy measure.
# Keeping it out of the general interaction chart prevents the severe
# Barnet status discontinuity from being mistaken for a substantive result.
interaction_plot_outcomes = [
    label for label in outcomes if label != "OpenLocal occupation-based vacancy proxy"
]
fig, axes = plt.subplots(1, 3, figsize=(10.2, 3.8))
interaction_colors = {"Absolute weighted flow": PALETTE["orange"], "Normalised weighted share": PALETTE["blue"]}
for ax, outcome_label in zip(axes, interaction_plot_outcomes):
    sub = interaction_results[interaction_results["outcome_label"].eq(outcome_label)]
    for y, predictor_label in enumerate(interaction_predictors):
        row = sub[sub["predictor_label"].eq(predictor_label)].iloc[0]
        ax.errorbar(
            row["interaction_beta"], y,
            xerr=[[row["interaction_beta"] - row["interaction_ci_low"]], [row["interaction_ci_high"] - row["interaction_beta"]]],
            fmt="o", color=interaction_colors[predictor_label], capsize=3, markersize=5,
        )
    ax.axvline(0, color="#8d9497", linewidth=0.9)
    ax.set_yticks([0, 1], ["Absolute", "Normalised"] if ax is axes[0] else ["", ""], fontsize=7)
    ax.set_title(outcome_label, fontsize=9.5)
    ax.set_xlabel("Exposure x distance\ncoefficient (95% CI)", fontsize=8)
    ax.grid(axis="x", alpha=0.18)
fig.suptitle("Does Commuting Distance Moderate the Residential Exposure Association?", x=0.02, ha="left", fontsize=13)
fig.tight_layout(rect=[0, 0, 1, 0.91])
fig.savefig(FIG_DIR / "fig_03_h1_exposure_distance_interactions.png", bbox_inches="tight", pad_inches=0.04, dpi=300)
plt.show()

# Vacancy-only sensitivity: remove LADs with a severe quarterly status discontinuity.
vacancy_sensitivity_rows = []
vacancy_outcome = outcomes["OpenLocal occupation-based vacancy proxy"]
for sample_label, sample in {
    "All eligible MSOAs": model_panel,
    "Exclude severe temporal anomaly LADs": model_panel[~model_panel["vacancy_temporal_anomaly_flag"]],
}.items():
    for predictor_label, predictor in predictors.items():
        row, _ = fit_refined_model(sample, vacancy_outcome, predictor, "Controlled")
        row.update({"sample": sample_label, "predictor_label": predictor_label})
        vacancy_sensitivity_rows.append(row)
vacancy_sensitivity = pd.DataFrame(vacancy_sensitivity_rows)
vacancy_sensitivity.to_csv(OUT_DIR / "h1_vacancy_anomaly_sensitivity.csv", index=False)

vacancy_interaction_sensitivity_rows = []
for sample_label, sample in {
    "All eligible MSOAs": model_panel,
    "Exclude severe temporal anomaly LADs": model_panel[~model_panel["vacancy_temporal_anomaly_flag"]],
}.items():
    for predictor_label, predictor in interaction_predictors.items():
        row, _ = fit_distance_interaction(sample, vacancy_outcome, predictor, predictor_label)
        row.update({"sample": sample_label, "outcome_label": "OpenLocal occupation-based vacancy proxy"})
        vacancy_interaction_sensitivity_rows.append(row)
vacancy_interaction_sensitivity = pd.DataFrame(vacancy_interaction_sensitivity_rows)
vacancy_interaction_sensitivity.to_csv(OUT_DIR / "h1_vacancy_distance_interaction_sensitivity.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(9.6, 3.8))
sensitivity_colors = {"All eligible MSOAs": PALETTE["grey"], "Exclude severe temporal anomaly LADs": PALETTE["blue_dark"]}
for y, predictor_label in enumerate(predictors):
    for offset, sample_label in [(-0.12, "All eligible MSOAs"), (0.12, "Exclude severe temporal anomaly LADs")]:
        row = vacancy_sensitivity[
            vacancy_sensitivity["predictor_label"].eq(predictor_label)
            & vacancy_sensitivity["sample"].eq(sample_label)
        ].iloc[0]
        axes[0].errorbar(
            row["beta_standardised_exposure"], y + offset,
            xerr=1.96 * row["robust_se"], fmt="o", color=sensitivity_colors[sample_label], capsize=3,
        )
axes[0].set_yticks(range(len(predictors)), ["Absolute", "Normalised", "Selected share"])
axes[0].set_title("A. Vacancy exposure coefficient")
axes[0].set_xlabel("Coefficient (95% CI)")
axes[0].axvline(0, color="#8d9497", linewidth=0.9)
axes[0].grid(axis="x", alpha=0.18)

for y, predictor_label in enumerate(interaction_predictors):
    for offset, sample_label in [(-0.12, "All eligible MSOAs"), (0.12, "Exclude severe temporal anomaly LADs")]:
        row = vacancy_interaction_sensitivity[
            vacancy_interaction_sensitivity["predictor_label"].eq(predictor_label)
            & vacancy_interaction_sensitivity["sample"].eq(sample_label)
        ].iloc[0]
        axes[1].errorbar(
            row["interaction_beta"], y + offset,
            xerr=1.96 * row["interaction_se"], fmt="o", color=sensitivity_colors[sample_label], capsize=3,
        )
axes[1].set_yticks(range(len(interaction_predictors)), ["Absolute", "Normalised"])
axes[1].set_title("B. Exposure x distance coefficient")
axes[1].set_xlabel("Interaction coefficient (95% CI)")
axes[1].axvline(0, color="#8d9497", linewidth=0.9)
axes[1].grid(axis="x", alpha=0.18)
legend_handles = [Line2D([0], [0], marker="o", color="none", markerfacecolor=color, markeredgecolor=color, label=label) for label, color in sensitivity_colors.items()]
fig.legend(handles=legend_handles, loc="lower center", ncol=2, frameon=False, fontsize=8)
fig.suptitle("OpenLocal Vacancy Sensitivity to the Severe Temporal Discontinuity", x=0.02, ha="left", fontsize=13)
fig.tight_layout(rect=[0, 0.12, 1, 0.91])
fig.savefig(FIG_DIR / "fig_04_h1_vacancy_anomaly_sensitivity.png", bbox_inches="tight", pad_inches=0.04, dpi=300)
plt.show()

display(interaction_results)
display(vacancy_sensitivity[["sample", "predictor_label", "n_obs", "beta_standardised_exposure", "p_value", "r_squared"]])
display(vacancy_interaction_sensitivity[["sample", "predictor_label", "n_obs", "interaction_beta", "interaction_p", "r_squared"]])

## 6. Residual Moran's I for the Primary Normalised Exposure-Distance Models

In [ ]:
def queen_edges(gdf):
    gdf = gdf.reset_index(drop=True)
    left, right = [], []
    sindex = gdf.sindex
    for i, geom in enumerate(gdf.geometry):
        if geom is None or geom.is_empty:
            continue
        candidates = sindex.query(geom, predicate="touches")
        for j in candidates:
            if i != j:
                left.append(i)
                right.append(int(j))
    return np.asarray(left, dtype=int), np.asarray(right, dtype=int)

def permutation_morans_i(values, edge_i, edge_j, permutations=499, seed=42):
    values = np.asarray(values, dtype=float)
    z = values - values.mean()
    denom = np.sum(z ** 2)
    n = len(z)
    s0 = len(edge_i)
    if n < 4 or denom == 0 or s0 == 0:
        return np.nan, np.nan, np.nan, n, s0
    observed = (n / s0) * (np.sum(z[edge_i] * z[edge_j]) / denom)
    rng = np.random.default_rng(seed)
    sims = np.empty(permutations)
    for p in range(permutations):
        zp = rng.permutation(z)
        sims[p] = (n / s0) * (np.sum(zp[edge_i] * zp[edge_j]) / denom)
    p_value = (np.sum(np.abs(sims) >= abs(observed)) + 1) / (permutations + 1)
    z_score = (observed - sims.mean()) / sims.std(ddof=1)
    return observed, z_score, p_value, n, s0

moran_rows = []
if len(residual_panel):
    for outcome_label, sub in residual_panel.groupby("outcome_label"):
        residual_gdf = boundaries[["MSOA21CD", "geometry"]].merge(sub, on="MSOA21CD", how="inner")
        edge_i, edge_j = queen_edges(residual_gdf)
        moran_i, z_value, p_value, n, links = permutation_morans_i(
            residual_gdf["residual"], edge_i, edge_j
        )
        moran_rows.append({
            "outcome": outcome_label,
            "moran_i": moran_i,
            "permutation_z": z_value,
            "permutation_p": p_value,
            "msoas": n,
            "directed_neighbor_links": links,
            "decision_note": "Consider spatial specification" if p_value < 0.05 else "No residual spatial model triggered",
        })
moran_results = pd.DataFrame(moran_rows)
moran_results.to_csv(OUT_DIR / "h1_primary_model_residual_moran.csv", index=False)
display(moran_results)

## 7. Remaining Work and Final Reporting Rule

1. Manually inspect the restricted premises candidates already selected by the postcode drill-down. Restricted addresses must not be exported to the public repository.
2. Confirm each manually reviewed candidate, where possible, as non-public commercial use, survey-timing difference, administrative lag, coverage difference or unresolved.
3. Report Green Street and OpenLocal as complementary outcome families. Correlation and gap statistics are descriptive diagnostics, not a formal reliability validation.
4. If residual Moran's I remains significant after the distance interaction, compare the primary OLS specification with an explicitly justified spatial-error or spatial-lag model. Do not add a spatial model solely because it is available.

The outputs produced here are refinement diagnostics. The final dissertation tables should report a deliberately selected subset of models rather than every specification generated during development.